# Nemotron-3-Nano LoRA SFT with CoT-Selected Training Data

## Approach

This notebook fine-tunes **Nemotron-3-Nano-30B-A3B-BF16** with LoRA using carefully curated Chain-of-Thought (CoT) training data.

The key insight is that **training data quality matters more than quantity**: rather than using all 9,000 training samples, I generated CoT reasoning via LLM inference and then **kept only the correct answers** verified by rule-based scoring.

### My Previous Notebooks

- [Doc-to-LoRA Knowledge Injection](https://www.kaggle.com/code/konbu17/doc-to-lora-knowledge-injection-nemotron-3-nano) — where I explored knowledge distillation for the 77-word dictionary memorization (Text Encryption) and multi-step SFT
- [Bit Manipulation Solver CoT Generator](https://www.kaggle.com/code/konbu17/bit-manipulation-solver-cot-generator) — where I developed per-bit boolean function analysis for the Bit Manipulation puzzles

### CoT Data Generation Pipeline

The training data (`train_split_with_cot.csv`) was generated by the following steps:

1. **Prompt construction**: Each puzzle prompt was augmented with the Kaggle official suffix: `Please put your final answer inside \boxed{}. For example: \boxed{your answer}`

2. **Type-specific prompt engineering**:
   - **Text Encryption**: Embedded the 77-word Alice's Wonderland dictionary into the prompt. Since these puzzles use monoalphabetic substitution ciphers with a fixed vocabulary, providing the encrypted→plain mapping for all 77 words (with `?` for unknown characters) dramatically improved accuracy.
   - **Bit Manipulation**: Embedded per-bit boolean function candidate analysis. Each output bit is independently determined by 1-3 input bits via boolean functions (ID 40%, AND 15%, XOR 8%, etc.). The solver enumerates all candidate functions for each bit and identifies CERTAIN vs AMBIGUOUS bits.
   - **Other types** (Gravitational Constant, Unit Conversion, Numeral Conversion, Equation Transformation): Standard step-by-step reasoning prompts.

3. **LLM inference** with temperature=0.7 to generate CoT reasoning

4. **Rule-based correctness filtering**: Extracted the final answer from `\boxed{}` and compared with ground truth using type-specific checkers:
   - Bit Manipulation: exact 8-bit binary match
   - Gravitational Constant / Unit Conversion: numeric match within ±0.05 absolute or ±0.5% relative error
   - Numeral Conversion: Roman numeral exact match (case-insensitive)
   - Text Encryption: case-insensitive exact match
   - Equation Transformation: exact string match
   
   **Only samples where the LLM's answer was verified correct were kept.**

5. **Token length control**: CoT outputs were checked against the Nemotron-3-Nano tokenizer. Any `generated_cot` exceeding **7,600 tokens** was sent back to the LLM with instructions to shorten by the overflow percentage while preserving the answer in `\boxed{}`. The limit of 7,600 (not 7,680) leaves headroom for the `</think>\n\boxed{answer}` suffix appended during training.

### Dataset Statistics

The resulting dataset contains **6,558 verified-correct CoT samples** out of 9,000 total:

| Puzzle Type | Total Available | Sampled for SFT | % Used | Note |
|---|---:|---:|---:|---|
| Gravitational Constant | 1,511 | 400 | 26.5% | Easy: 99.7% pass rate |
| Numeral Conversion | 1,491 | 300 | 20.1% | Easy: 99.6% pass rate |
| Unit Conversion | 1,342 | 700 | 52.2% | Good: 88.7% pass rate |
| Text Encryption | 1,407 | 700 | 49.8% | Good: 94.3% pass rate (with 77-word dict prompt) |
| Bit Manipulation | 607 | 607 | **100%** | Hard: only 40.3% pass rate — all correct samples used |
| Equation Transformation | 200 | 200 | **100%** | Hardest: only 13.6% pass rate — all correct samples used |
| **Total** | **6,558** | **2,907** | **44.3%** | |

Bit Manipulation and Equation Transformation had much lower pass rates because these puzzle types require complex pattern recognition that even strong LLMs struggle with. All verified-correct samples were used for these types.

### What I Did NOT Tune

My main effort was on **data preparation** (CoT generation + correctness filtering). I did **not** extensively tune the LoRA configuration — I used the standard competition target modules (`in_proj|out_proj|up_proj|down_proj`) with rank=32, alpha=32. If you want to improve accuracy further, tuning LoRA hyperparameters (alpha, dropout, learning rate, epochs) or trying different target module selections could help!

Oops, I forgot to say that I didn't input solvers into Equation Transformation, so we may get better scores from inputting "Equation Transformation" knowledge to solve.
(I've not yet analyzed "Equation Transformation" thingy...)

---

## Mode Selection

Set **one** of these to `1` to choose your submission mode:

In [1]:
# ============================================================
# MODE SELECTION — set exactly one to 1
# ============================================================

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})

{'TRAIN_ON_KAGGLE': 1, 'USE_PRETRAINED': 0, 'PRETRAINED_ADAPTER_DATASET_PATH': '/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection'}


## Setup & Model Loading

In [2]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps",
        "--target", target,
        "--upgrade",
        "--ignore-installed",
        wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)

site.addsitedir(target)

print("Custom target added:", target)

import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

Found Triton wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Custom target added: /kaggle/working/pydeps
triton spec: ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7f8148298500>, origin='/kaggle/working/pydeps/triton/__init__.py', submodule_search_locations=['/kaggle/working/pydeps/triton'])


In [3]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    # Add utility script to Python path (provides helper binaries)
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    # Copy ptxas-blackwell to /tmp with execute permissions
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")


Training environment fixes applied.


In [4]:
# Install trl if needed (for training mode)
if TRAIN_ON_KAGGLE:
    try:
        import trl
        print(f"trl already installed: {trl.__version__}")
    except ImportError:
        # Try offline install first, then online
        offline_path = "/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/"
        if os.path.exists(offline_path):
            subprocess.run(f"pip install --no-index --find-links={offline_path} trl", shell=True)
        else:
            subprocess.run("pip install trl", shell=True)
        import trl
        print(f"trl installed: {trl.__version__}")

Looking in links: /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/
Processing /kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/trl-0.29.1-py3-none-any.whl
trl installed: 0.29.1


In [5]:
if TRAIN_ON_KAGGLE:
    import glob, importlib.util, os, subprocess, sys, types

    def sh(cmd: str, check: bool = True):
        print("+", cmd)
        return subprocess.run(cmd, shell=True, check=check)

    def find_spec(name: str) -> bool:
        return importlib.util.find_spec(name) is not None

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    all_mamba = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")
    all_datasets = recursive_wheels("datasets-*.whl")
    all_trl = recursive_wheels("trl-*.whl")
    all_multiprocess = recursive_wheels("multiprocess-*.whl")
    all_dill = recursive_wheels("dill-*.whl")
    all_xxhash = recursive_wheels("xxhash-*.whl")

    print("Found mamba wheels:", all_mamba)
    print("Found causal-conv1d wheels:", all_causal)

    import torch
    py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
    torch_mm = ".".join(torch.__version__.split("+")[0].split(".")[:2])
    abi_tag = "cxx11abiTRUE" if torch.compiled_with_cxx11_abi() else "cxx11abiFALSE"

    print("Python:", sys.version)
    print("Torch: ", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Torch CUDA:", torch.version.cuda)
    print("Wheel selector:", {"py_tag": py_tag, "torch": torch_mm, "abi": abi_tag})

    if not torch.cuda.is_available():
        raise RuntimeError(
            "TRAIN_ON_KAGGLE=1 requires GPU runtime because mamba_ssm wheel is CUDA-based."
        )

    def pick_best(wheels):
        exact = [w for w in wheels if py_tag in w and f"torch{torch_mm}" in w and abi_tag in w]
        if exact:
            return exact[-1]
        py_only = [w for w in wheels if py_tag in w]
        if py_only:
            return py_only[-1]
        return None

    if not find_spec("datasets"):
        w = pick_best(all_datasets)
        if w:
            sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')
    if not find_spec("trl"):
        w = pick_best(all_trl)
        if w:
            sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"')
    for pkg, wheels in [("multiprocess", all_multiprocess), ("dill", all_dill), ("xxhash", all_xxhash)]:
        if not find_spec(pkg):
            w = pick_best(wheels)
            if w:
                sh(f'{sys.executable} -m pip install --no-index --no-deps "{w}"', check=False)

    if not find_spec("mamba_ssm"):
        causal_wheel = pick_best(all_causal)
        mamba_wheel = pick_best(all_mamba)

        print("Selected causal wheel:", causal_wheel)
        print("Selected mamba wheel:", mamba_wheel)

        if causal_wheel:
            sh(f'{sys.executable} -m pip install --no-index --no-deps "{causal_wheel}"')
        if mamba_wheel:
            sh(f'{sys.executable} -m pip install --no-index --no-deps "{mamba_wheel}"')
        else:
            raise FileNotFoundError(
                f"No compatible mamba_ssm wheel found under /kaggle/input for "
                f"py={py_tag}, torch={torch_mm}, abi={abi_tag}."
            )

    import datasets
    import trl

    for _mod_name in [
        'mamba_ssm.modules.mamba3',
        'mamba_ssm.ops.cute',
        'mamba_ssm.ops.cute.mamba3',
        'mamba_ssm.ops.cute.mamba3.mamba3_step_fn',
    ]:
        sys.modules[_mod_name] = types.ModuleType(_mod_name)
    sys.modules['mamba_ssm.modules.mamba3'].Mamba3 = None

    import mamba_ssm

    print(f'datasets:  {datasets.__version__}')
    print(f'trl:       {trl.__version__}')
    print(f'mamba_ssm: {mamba_ssm.__version__}')
else:
    print("USE_PRETRAINED=1: skipping datasets / trl / mamba_ssm installation.")


Found mamba wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl']
Found causal-conv1d wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl']
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch:  2.10.0+cu128
CUDA available: True
Torch CUDA: 12.8
Wheel selector: {'py_tag': 'cp312', 'torch': '2.10', 'abi': 'cxx11abiTRUE'}
Selected causal wheel: /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Selected mamba wheel: /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
+ /usr/bin/python3 -m pip install --no-index --no-deps "/kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
Processing /kaggle/input/datase

In [6]:
if TRAIN_ON_KAGGLE:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import kagglehub

    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16,
    )
    print("Model loaded.")
else:
    print("USE_PRETRAINED=1: skipping base model/tokenizer loading.")


Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded.


In [7]:
if TRAIN_ON_KAGGLE:
    from peft import LoraConfig, get_peft_model, TaskType

    LORA_RANK = 32
    LORA_ALPHA = 32

    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping PEFT model construction.")


trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


## Mode A: Train on Kaggle

In [8]:
import pandas as pd
DATASET_PATH = "/kaggle/input/datasets/kishanvavdara/nemotron-reasoning-traj/nemotron_traj.csv"
df = pd.read_csv(DATASET_PATH)
df = df[df['correctness']=='true']
df

,id,prompt,generated,correct answer,generated answer,correctness,problem type
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",We need to deduce the numeral system used. Giv...,XXXVIII,XXXVIII,true,numeral
10,0047365c,"In Alice's Wonderland, a secret unit conversio...",We need to infer pattern: mapping from meters ...,10.62,10.61,true,unit_conversion
12,0059df78,"In Alice's Wonderland, a secret unit conversio...",We have mapping of original length to converte...,43.43,43.43,true,unit_conversion
15,00619cba,"In Alice's Wonderland, the gravitational const...",We have observations of distance d given times...,91.84,91.83,true,gravity
18,006a46d3,"In Alice's Wonderland, a secret unit conversio...",We need to find conversion rule. Given inputs ...,19.00,19.00,true,unit_conversion
...,...,...,...,...,...,...,...
9491,ffa81415,"In Alice's Wonderland, a secret unit conversio...",We need to infer the conversion from examples....,18.53,18.54,true,unit_conversion
9492,ffaf05c5,"In Alice's Wonderland, numbers are secretly co...",We need to infer pattern from examples: 52 -> ...,XLVIII,XLVIII,true,numeral
9493,ffb06ece,"In Alice's Wonderland, a secret unit conversio...",We need to find the transformation rule. Given...,18.47,18.46,true,unit_conversion
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",We need to infer the conversion function mappi...,32.45,32.45,true,unit_conversion


In [9]:
mapping = {
    "numeral": "Numeral Conversion",
    "gravity": "Gravitational Constant",
    "unit_conversion": "Unit Conversion",
    "cipher": "Text Encryption",
    "bit_manipulation": "Bit Manipulation",
    "equation_numeric": "Equation Transformation",
    "equation_symbolic": "Equation Transformation",
}

df["problem type"] = df["problem type"].map(mapping)

In [10]:
df = df.rename(columns={
    "problem type": "type",
    "correct answer": "answer",
    "generated": "generated_cot",
})
df

,id,prompt,generated_cot,answer,generated answer,correctness,type
3,001b24c4,"In Alice's Wonderland, numbers are secretly co...",We need to deduce the numeral system used. Giv...,XXXVIII,XXXVIII,true,Numeral Conversion
10,0047365c,"In Alice's Wonderland, a secret unit conversio...",We need to infer pattern: mapping from meters ...,10.62,10.61,true,Unit Conversion
12,0059df78,"In Alice's Wonderland, a secret unit conversio...",We have mapping of original length to converte...,43.43,43.43,true,Unit Conversion
15,00619cba,"In Alice's Wonderland, the gravitational const...",We have observations of distance d given times...,91.84,91.83,true,Gravitational Constant
18,006a46d3,"In Alice's Wonderland, a secret unit conversio...",We need to find conversion rule. Given inputs ...,19.00,19.00,true,Unit Conversion
...,...,...,...,...,...,...,...
9491,ffa81415,"In Alice's Wonderland, a secret unit conversio...",We need to infer the conversion from examples....,18.53,18.54,true,Unit Conversion
9492,ffaf05c5,"In Alice's Wonderland, numbers are secretly co...",We need to infer pattern from examples: 52 -> ...,XLVIII,XLVIII,true,Numeral Conversion
9493,ffb06ece,"In Alice's Wonderland, a secret unit conversio...",We need to find the transformation rule. Given...,18.47,18.46,true,Unit Conversion
9496,ffd5bada,"In Alice's Wonderland, a secret unit conversio...",We need to infer the conversion function mappi...,32.45,32.45,true,Unit Conversion


In [11]:
if TRAIN_ON_KAGGLE:
    import pandas as pd
    import random
    import gc, time
    from datasets import Dataset as HFDataset
    from trl import SFTTrainer, SFTConfig

    SEED = 123
    PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

    # --- Dataset path ---
    DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection/train_split_with_cot.csv"
    df = pd.read_csv(DATASET_PATH)
    print(f"Full dataset: {len(df)} rows")
    print(df["type"].value_counts().sort_index())

    # --- Type-based sampling ---
    TYPE_SAMPLES = {
        "Numeral Conversion": 300,
        "Gravitational Constant": 400,
        "Unit Conversion": 700,
        "Text Encryption": 700,
        "Bit Manipulation": 607,        # all available
        "Equation Transformation": 200,  # all available
    }

    sampled_dfs = []
    for ptype, n_samples in TYPE_SAMPLES.items():
        subset = df[df["type"] == ptype]
        if n_samples >= len(subset):
            sampled = subset
        else:
            sampled = subset.sample(n=n_samples, random_state=SEED)
        print(f"  {ptype}: {len(subset)} -> {len(sampled)}")
        sampled_dfs.append(sampled)

    train_df = pd.concat(sampled_dfs, ignore_index=True)
    train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"\nTraining samples: {len(train_df)}")

    # --- Build SFT dataset ---
    import re
    records = []
    for _, row in train_df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        cot = str(row["generated_cot"])
        if not cot or cot == "nan" or len(cot.strip()) < 5:
            continue
        cot_cleaned = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
        user_content = prompt + PROMPT_SUFFIX
        # chat template auto-adds <think>\n, so assistant starts with CoT directly
        assistant_content = cot_cleaned + f"\n</think>\n\\boxed{{{answer}}}"
        records.append({"messages": [
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]})
    dataset = HFDataset.from_list(records)
    print(f"SFT records: {len(records)}")

    # --- Training ---
    training_args = SFTConfig(
        output_dir="/kaggle/working/sft_output",
        num_train_epochs=2, # 1 # 2
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4, # 5e-5 # 1e-4
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        max_length=4096, # 7680 # 4096
        logging_steps=10,
        save_strategy="no",
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=SEED,
        report_to="none",
        packing=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        processing_class=tokenizer,
    )

    print("Starting SFT training...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"Training done in {elapsed/60:.1f} min")

    # Save adapter
    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to {ADAPTER_DIR}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Full dataset: 6558 rows
type
Bit Manipulation            607
Equation Transformation     200
Gravitational Constant     1511
Numeral Conversion         1491
Text Encryption            1407
Unit Conversion            1342
Name: count, dtype: int64
  Numeral Conversion: 1491 -> 300
  Gravitational Constant: 1511 -> 400
  Unit Conversion: 1342 -> 700
  Text Encryption: 1407 -> 700
  Bit Manipulation: 607 -> 607
  Equation Transformation: 200 -> 200

Training samples: 2907
SFT records: 2907


Tokenizing train dataset:   0%|          | 0/2907 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2907 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


Starting SFT training...


Step,Training Loss
10,10.753908
20,9.166288
30,7.529663
40,5.024210
50,4.735238
60,3.277515
70,3.524226
80,3.256723
90,3.051063
100,3.399253


Training done in 406.1 min
Adapter saved to /kaggle/working/sft_adapter


## Mode B: Load Pre-trained LoRA

In [12]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")


TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.


## Create submission.zip

In [13]:
import json, os, shutil, zipfile

OUTPUT_DIR = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging freshly trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter directly from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"] = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done! Ready to submit.")


Packaging freshly trained adapter from: /kaggle/working/sft_adapter
Copied adapter_config.json (0.0 MB)
Copied adapter_model.safetensors (3359.2 MB)
  Added adapter_config.json
  Added adapter_model.safetensors

submission.zip: 3084.5 MB
Done! Ready to submit.
